# Chapter 01: Triangles

**Source span:** printed pages 3-25; PDF pages 21-43. The source pages were read as private scanned page renders for orientation only. This notebook is original teaching material: it does not reuse textbook prose, exercises, screenshots, page images, or figure layouts.

**Chapter goal.** A triangle is a small object, but this chapter uses it as a laboratory for Euclidean geometry. By the end of the notebook you should be able to translate classical triangle statements into inspectable computations: congruent pieces become distance and angle equalities, concurrence becomes a determinant or line intersection, circle theorems become equal radii or equal side distances, and extremum problems become constrained minimizations with geometric witnesses.

## Computational Translation Guide

| Source idea | Computational model | What to inspect |
| --- | --- | --- |
| Primitive concepts and congruence | points as coordinate pairs, lines as equations, reflection as an isometry | distances and angles before and after reflection |
| Medians and centroid | midpoint functions and barycentric average | the medians share one point; the ratio on each median is 2:1 |
| Incenter and circumcenter | side-length weighted average; perpendicular-bisector intersection | equal distances to sides; equal distances to vertices |
| Euler line | circumcenter `O`, centroid `G`, orthocenter `H` | the vector identity `H - G = 2(G - O)` |
| Nine-point circle | side midpoints, altitude feet, and midpoints from vertices to `H` | all nine points lie on a circle of radius `R/2` centered at `(O+H)/2` |
| Extremum problems | orthic triangle and Fermat point checked by optimization | a geometric candidate agrees with a numerical minimum |
| Morley-style trisectors | intersections of adjacent angle trisectors | the trisector triangle is equilateral |

## Compact Visual Storyboard Implemented

1. Reflection exposes primitive congruence and the isosceles base-angle move.
2. Medians expose the centroid as a barycentric average.
3. The incenter and circumcenter are drawn together to separate side-distance and vertex-distance centers.
4. The Euler line and nine-point circle are checked in one triangle and in a Plotly vertex-motion experiment.
5. Fagnano's orthic triangle and Fermat's 120-degree point are compared with numerical optimization.
6. Morley's adjacent trisectors are constructed and measured.
7. A proof-dependency graph, exact SymPy identities, artifact sizes, and final JSON checks make the notebook auditable.

In [ ]:
from __future__ import annotations

from pathlib import Path
import math
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import Arc, Circle, Polygon
import networkx as nx
import numpy as np
import plotly.graph_objects as go
from scipy.optimize import minimize
import sympy as sp

CHAPTER_NO = 1
HERE = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [HERE, *HERE.parents]:
    if (candidate / "00-book-index.ipynb").exists() and (candidate / "utils").exists():
        BOOK_ROOT = candidate
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not locate the Introduction-to-Geometry book root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from IPython.display import Markdown, display
from utils.artifacts import assert_artifacts, book_relative, display_artifact, ensure_chapter_artifact_dirs, write_csv, write_json
from utils.course import chapter_by_no

chapter = chapter_by_no(CHAPTER_NO)
ARTIFACT_DIRS = ensure_chapter_artifact_dirs(CHAPTER_NO, BOOK_ROOT)
for legacy_name in ["concept_configuration.svg", "parameter_experiment.svg"]:
    legacy_path = ARTIFACT_DIRS["figures"] / legacy_name
    if legacy_path.exists():
        legacy_path.unlink()

def artifact(kind: str, name: str) -> Path:
    return ARTIFACT_DIRS[kind] / name

def rel(path: Path) -> str:
    return book_relative(path, BOOK_ROOT)

display(Markdown(f"Loaded **{chapter['title']}**. Artifacts write under `{rel(ARTIFACT_DIRS['figures'].parent)}`."))

## How To Read The Visuals

Each visual has a specific job. The reflection figure is a preservation test: look for pairs of segments or angles that should remain equal after a mirror move. The centroid figure is an affine test: the exact shape of the triangle can change, but midpoint and average operations should still force the same 2:1 median ratios. The incenter/circumcenter figure is a contrast test: one center is controlled by side lines and tangent distances, while the other is controlled by vertex distances and a surrounding circle. The Euler and nine-point visuals are invariant tests: a line, a midpoint, and a radius relation survive when the triangle is varied. The extremum figure is an optimization test: a construction is credible only after its perimeter or distance-sum claim agrees with a numerical minimizer. The Morley figure is a surprise test: trisectors look unstable, so the equilateral result is measured rather than assumed.

When you rerun the notebook, read every artifact together with the check dictionary printed below it. The diagram tells you where to look; the numbers tell you whether the construction actually satisfied the stated relation. This pairing is the main computational habit of the chapter.

## Geometry Helpers

The notebook uses explicit coordinate constructions instead of a generic visual renderer. The helper functions are intentionally small: they implement distances, line intersections, reflections, triangle centers, and the special constructions used later. The same functions feed the drawings and the checks, so a mislabeled diagram is likely to fail a numeric invariant.

In [ ]:
PALETTE = {
    "ink": "#1f2937", "blue": "#2563eb", "green": "#059669", "orange": "#d97706",
    "red": "#dc2626", "purple": "#7c3aed", "gray": "#6b7280", "gold": "#f59e0b",
}
BASE_TRIANGLE = np.array([[0.15, 2.70], [-2.20, 0.05], [2.35, 0.15]], dtype=float)
A, B, C = BASE_TRIANGLE

def cross2(u: np.ndarray, v: np.ndarray) -> float:
    return float(u[0] * v[1] - u[1] * v[0])

def norm(v: np.ndarray) -> float:
    return float(np.linalg.norm(v))

def rotate(v: np.ndarray, angle: float) -> np.ndarray:
    c, s = math.cos(angle), math.sin(angle)
    return np.array([c * v[0] - s * v[1], s * v[0] + c * v[1]], dtype=float)

def angle_between(u: np.ndarray, v: np.ndarray) -> float:
    value = float(np.dot(u, v) / (norm(u) * norm(v)))
    return math.acos(max(-1.0, min(1.0, value)))

def angle_ccw(u: np.ndarray, v: np.ndarray) -> float:
    start = math.atan2(u[1], u[0]); end = math.atan2(v[1], v[0])
    return (end - start) % (2 * math.pi)

def triangle_area(tri: np.ndarray) -> float:
    return 0.5 * cross2(tri[1] - tri[0], tri[2] - tri[0])

def side_lengths(tri: np.ndarray) -> tuple[float, float, float]:
    A0, B0, C0 = tri
    return norm(B0 - C0), norm(C0 - A0), norm(A0 - B0)

def midpoint(P: np.ndarray, Q: np.ndarray) -> np.ndarray:
    return 0.5 * (P + Q)

def line_intersection(P: np.ndarray, d: np.ndarray, Q: np.ndarray, e: np.ndarray) -> np.ndarray:
    denom = cross2(d, e)
    if abs(denom) < 1e-12:
        raise ValueError("nearly parallel lines")
    return P + cross2(Q - P, e) / denom * d

def foot(P: np.ndarray, Q: np.ndarray, R: np.ndarray) -> np.ndarray:
    v = R - Q
    return Q + float(np.dot(P - Q, v) / np.dot(v, v)) * v

def point_line_distance(P: np.ndarray, Q: np.ndarray, R: np.ndarray) -> float:
    return abs(cross2(R - Q, P - Q)) / norm(R - Q)

def reflect_point(P: np.ndarray, Q: np.ndarray, R: np.ndarray) -> np.ndarray:
    v = R - Q
    projection = Q + float(np.dot(P - Q, v) / np.dot(v, v)) * v
    return 2 * projection - P

def centroid(tri: np.ndarray) -> np.ndarray:
    return tri.mean(axis=0)

def incenter(tri: np.ndarray) -> tuple[np.ndarray, float]:
    A0, B0, C0 = tri; a, b, c = side_lengths(tri)
    I = (a * A0 + b * B0 + c * C0) / (a + b + c)
    return I, point_line_distance(I, B0, C0)

def circumcenter(tri: np.ndarray) -> tuple[np.ndarray, float]:
    A0, B0, C0 = tri
    ax, ay = A0; bx, by = B0; cx, cy = C0
    d = 2 * (ax * (by - cy) + bx * (cy - ay) + cx * (ay - by))
    if abs(d) < 1e-12:
        raise ValueError("degenerate triangle")
    ux = ((ax*ax+ay*ay)*(by-cy) + (bx*bx+by*by)*(cy-ay) + (cx*cx+cy*cy)*(ay-by)) / d
    uy = ((ax*ax+ay*ay)*(cx-bx) + (bx*bx+by*by)*(ax-cx) + (cx*cx+cy*cy)*(bx-ax)) / d
    O = np.array([ux, uy], dtype=float)
    return O, norm(O - A0)

def orthocenter(tri: np.ndarray) -> np.ndarray:
    O, _ = circumcenter(tri)
    return tri[0] + tri[1] + tri[2] - 2 * O

def nine_point_data(tri: np.ndarray) -> dict[str, object]:
    A0, B0, C0 = tri; O, R = circumcenter(tri); H = orthocenter(tri); N = midpoint(O, H)
    points = {
        "mid_BC": midpoint(B0, C0), "mid_CA": midpoint(C0, A0), "mid_AB": midpoint(A0, B0),
        "foot_A": foot(A0, B0, C0), "foot_B": foot(B0, C0, A0), "foot_C": foot(C0, A0, B0),
        "mid_AH": midpoint(A0, H), "mid_BH": midpoint(B0, H), "mid_CH": midpoint(C0, H),
    }
    residuals = {name: abs(norm(P - N) - R / 2) for name, P in points.items()}
    return {"O": O, "R": R, "H": H, "N": N, "points": points, "residuals": residuals}

assert triangle_area(BASE_TRIANGLE) > 0

In [ ]:
def draw_clean_axes(ax, title: str | None = None) -> None:
    ax.set_aspect("equal", adjustable="box"); ax.set_xticks([]); ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)
    if title:
        ax.set_title(title, fontsize=12, weight="bold", color=PALETTE["ink"])

def draw_triangle(ax, tri: np.ndarray, *, labels=("A", "B", "C"), color=None, lw=2.2, fill=False) -> None:
    color = color or PALETTE["ink"]
    poly = np.vstack([tri, tri[0]])
    if fill:
        ax.add_patch(Polygon(tri, closed=True, facecolor="#eff6ff", edgecolor="none", alpha=0.55))
    ax.plot(poly[:, 0], poly[:, 1], color=color, lw=lw)
    for P, label in zip(tri, labels):
        ax.scatter(P[0], P[1], s=38, color=color, zorder=5)
        ax.text(P[0] + 0.06, P[1] + 0.06, label, fontsize=10, color=color, weight="bold")

def add_circle(ax, center: np.ndarray, radius: float, *, color: str, lw=1.8, ls="-") -> None:
    ax.add_patch(Circle(center, radius, fill=False, edgecolor=color, lw=lw, ls=ls))

def add_angle_arc(ax, center: np.ndarray, P: np.ndarray, Q: np.ndarray, radius: float, color: str, label: str | None = None) -> None:
    a1 = math.degrees(math.atan2(*(P - center)[::-1])); a2 = math.degrees(math.atan2(*(Q - center)[::-1]))
    while a2 < a1:
        a2 += 360
    if a2 - a1 > 180:
        a1, a2 = a2, a1 + 360
    ax.add_patch(Arc(center, 2 * radius, 2 * radius, theta1=a1, theta2=a2, color=color, lw=1.5))
    if label:
        amid = math.radians((a1 + a2) / 2)
        ax.text(center[0] + 1.18 * radius * math.cos(amid), center[1] + 1.18 * radius * math.sin(amid), label, color=color, fontsize=9)

def equilateral_external(P: np.ndarray, Q: np.ndarray, opposite: np.ndarray) -> np.ndarray:
    cand1 = P + rotate(Q - P, math.pi / 3); cand2 = P + rotate(Q - P, -math.pi / 3)
    side_of_opposite = cross2(Q - P, opposite - P)
    return cand1 if cross2(Q - P, cand1 - P) * side_of_opposite < 0 else cand2

def fermat_point(tri: np.ndarray) -> tuple[np.ndarray, dict[str, np.ndarray]]:
    A0, B0, C0 = tri
    X_a = equilateral_external(B0, C0, A0); X_b = equilateral_external(C0, A0, B0); X_c = equilateral_external(A0, B0, C0)
    F = line_intersection(A0, X_a - A0, B0, X_b - B0)
    return F, {"X_a": X_a, "X_b": X_b, "X_c": X_c}

def morley_triangle(tri: np.ndarray) -> tuple[np.ndarray, dict[str, tuple[np.ndarray, np.ndarray]]]:
    vertices = list(tri); rays = []
    for i, V in enumerate(vertices):
        nxt = vertices[(i + 1) % 3]; prv = vertices[(i - 1) % 3]
        start = nxt - V; opening = angle_ccw(start, prv - V)
        rays.append((rotate(start, opening / 3), rotate(start, 2 * opening / 3)))
    dA1, dA2 = rays[0]; dB1, dB2 = rays[1]; dC1, dC2 = rays[2]
    P = line_intersection(vertices[1], dB1, vertices[2], dC2)
    Q = line_intersection(vertices[2], dC1, vertices[0], dA2)
    R = line_intersection(vertices[0], dA1, vertices[1], dB2)
    return np.array([P, Q, R]), {"A": (dA1, dA2), "B": (dB1, dB2), "C": (dC1, dC2)}

def save_figure(fig, path: Path) -> Path:
    path.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(path, dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return path

## Primitive Concepts And Congruence

The first computational move is deliberately modest: make an isometry visible. A reflection is a map from points to points that preserves distances and angles. In an isosceles-triangle proof, the mirror through the apex and midpoint of the base interchanges the two equal sides. The invariant is not the picture's symmetry alone; it is the measured equality of reflected lengths and base angles.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.8))
ax = axes[0]
draw_clean_axes(ax, "Reflection preserves metric data")
mirror_P = np.array([0.0, -0.25]); mirror_Q = np.array([0.0, 2.85])
tri_left = np.array([[-2.1, 2.05], [-2.65, 0.35], [-0.85, 0.65]])
tri_reflected = np.array([reflect_point(P, mirror_P, mirror_Q) for P in tri_left])
draw_triangle(ax, tri_left, labels=("P", "Q", "R"), color=PALETTE["blue"], lw=2.0, fill=True)
draw_triangle(ax, tri_reflected, labels=("P'", "Q'", "R'"), color=PALETTE["orange"], lw=2.0)
ax.plot([mirror_P[0], mirror_Q[0]], [mirror_P[1], mirror_Q[1]], color=PALETTE["gray"], lw=1.8, ls="--")
ax.text(0.08, 2.65, "mirror", color=PALETTE["gray"], fontsize=9)
for P, P_ref in zip(tri_left, tri_reflected):
    ax.plot([P[0], P_ref[0]], [P[1], P_ref[1]], color="#94a3b8", lw=0.9, ls=":")
ax.set_xlim(-3.0, 3.0); ax.set_ylim(-0.35, 3.1)

ax = axes[1]
draw_clean_axes(ax, "Isosceles triangle as a congruence test")
iso_A = np.array([0.0, 2.75]); iso_B = np.array([-1.75, 0.0]); iso_C = np.array([1.75, 0.0])
iso_tri = np.array([iso_A, iso_B, iso_C])
draw_triangle(ax, iso_tri, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
mid_BC_iso = midpoint(iso_B, iso_C)
ax.plot([iso_A[0], mid_BC_iso[0]], [iso_A[1], mid_BC_iso[1]], color=PALETTE["purple"], lw=1.8, ls="--")
ax.scatter(mid_BC_iso[0], mid_BC_iso[1], color=PALETTE["purple"], s=32, zorder=5)
ax.text(mid_BC_iso[0] + 0.07, mid_BC_iso[1] - 0.16, "M", color=PALETTE["purple"], fontsize=10, weight="bold")
add_angle_arc(ax, iso_B, iso_A, iso_C, 0.34, PALETTE["green"], "equal")
add_angle_arc(ax, iso_C, iso_B, iso_A, 0.34, PALETTE["green"], "equal")
ax.text(-1.06, 1.37, "AB", color=PALETTE["blue"], fontsize=9); ax.text(0.82, 1.37, "AC", color=PALETTE["orange"], fontsize=9)
ax.set_xlim(-2.35, 2.35); ax.set_ylim(-0.35, 3.15)
primitive_path = save_figure(fig, artifact("figures", "primitive-congruence-reflection.png"))

reflection_lengths = [norm(tri_left[(i + 1) % 3] - tri_left[i]) for i in range(3)]
reflected_lengths = [norm(tri_reflected[(i + 1) % 3] - tri_reflected[i]) for i in range(3)]
congruence_checks = {
    "max_reflected_side_length_error": float(max(abs(a - b) for a, b in zip(reflection_lengths, reflected_lengths))),
    "isosceles_side_error": float(abs(norm(iso_A - iso_B) - norm(iso_A - iso_C))),
    "isosceles_base_angle_error_degrees": float(abs(math.degrees(angle_between(iso_A - iso_B, iso_C - iso_B)) - math.degrees(angle_between(iso_B - iso_C, iso_A - iso_C)))),
}
assert congruence_checks["max_reflected_side_length_error"] < 1e-12
assert congruence_checks["isosceles_side_error"] < 1e-12
assert congruence_checks["isosceles_base_angle_error_degrees"] < 1e-12

display_artifact(primitive_path, width=760)
congruence_checks

## Medians, Centroid, Incenter, And Circumcenter

The median construction is affine: it uses only straightness, midpoint, and averaging. In coordinates, the centroid is the barycentric average `(A+B+C)/3`, so it has no reason to care about side lengths or angles. The incenter and circumcenter are metric centers. The incenter is equally far from the three side lines; the circumcenter is equally far from the three vertices. Seeing all three separately helps prevent the misconception that triangle centers are interchangeable.

In [ ]:
G = centroid(BASE_TRIANGLE)
M_a = midpoint(B, C); M_b = midpoint(C, A); M_c = midpoint(A, B)
fig, ax = plt.subplots(figsize=(6.7, 5.5))
draw_clean_axes(ax, "Medians meet at the centroid")
draw_triangle(ax, BASE_TRIANGLE, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
for V, M, name in [(A, M_a, "M_a"), (B, M_b, "M_b"), (C, M_c, "M_c")]:
    ax.plot([V[0], M[0]], [V[1], M[1]], color=PALETTE["blue"], lw=1.5, alpha=0.78)
    ax.scatter(M[0], M[1], color=PALETTE["blue"], s=34, zorder=5)
    ax.text(M[0] + 0.05, M[1] + 0.05, name, color=PALETTE["blue"], fontsize=9)
ax.scatter(G[0], G[1], color=PALETTE["red"], s=58, zorder=6)
ax.text(G[0] + 0.07, G[1] + 0.07, "G", color=PALETTE["red"], fontsize=11, weight="bold")
ax.text(-2.15, 2.45, "G = (A+B+C)/3", fontsize=10, color=PALETTE["ink"])
ax.set_xlim(-2.65, 2.7); ax.set_ylim(-0.35, 3.0)
centroid_path = save_figure(fig, artifact("figures", "centroid-medians-barycentric.png"))

median_ratios = {}
for label, V, M in [("A", A, M_a), ("B", B, M_b), ("C", C, M_c)]:
    median_ratios[f"{label}G_over_GM"] = float(norm(V - G) / norm(G - M))
    median_ratios[f"{label}_median_collinearity_area"] = float(abs(cross2(G - V, M - V)))
assert all(abs(value - 2.0) < 1e-12 for key, value in median_ratios.items() if key.endswith("over_GM"))
assert all(value < 1e-12 for key, value in median_ratios.items() if key.endswith("area"))

display_artifact(centroid_path, width=640)
median_ratios

In [ ]:
I, r = incenter(BASE_TRIANGLE)
O, R = circumcenter(BASE_TRIANGLE)
fig, ax = plt.subplots(figsize=(7.2, 5.8))
draw_clean_axes(ax, "Side-distance center versus vertex-distance center")
draw_triangle(ax, BASE_TRIANGLE, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
add_circle(ax, I, r, color=PALETTE["green"], lw=2.0)
add_circle(ax, O, R, color=PALETTE["purple"], lw=1.8, ls="--")
for V in BASE_TRIANGLE:
    ax.plot([V[0], I[0]], [V[1], I[1]], color=PALETTE["green"], lw=1.0, alpha=0.7)
for M in [M_a, M_b, M_c]:
    ax.plot([M[0], O[0]], [M[1], O[1]], color=PALETTE["purple"], lw=1.0, ls=":", alpha=0.75)
ax.scatter([I[0], O[0]], [I[1], O[1]], color=[PALETTE["green"], PALETTE["purple"]], s=[62, 62], zorder=6)
ax.text(I[0] + 0.06, I[1] - 0.16, "I", color=PALETTE["green"], fontsize=11, weight="bold")
ax.text(O[0] + 0.06, O[1] + 0.08, "O", color=PALETTE["purple"], fontsize=11, weight="bold")
ax.text(-2.55, 3.08, "I: equal distances to the three sides", color=PALETTE["green"], fontsize=9)
ax.text(-2.55, 2.86, "O: equal distances to the three vertices", color=PALETTE["purple"], fontsize=9)
ax.set_xlim(-3.0, 3.0); ax.set_ylim(-1.1, 3.45)
incenter_path = save_figure(fig, artifact("figures", "incenter-circumcenter-constructors.png"))

side_distance_errors = [abs(point_line_distance(I, B, C) - point_line_distance(I, C, A)), abs(point_line_distance(I, C, A) - point_line_distance(I, A, B))]
vertex_radius_errors = [abs(norm(O - A) - norm(O - B)), abs(norm(O - B) - norm(O - C))]
center_checks = {
    "incenter_max_side_distance_error": float(max(side_distance_errors)),
    "circumcenter_max_vertex_radius_error": float(max(vertex_radius_errors)),
    "inradius": float(r),
    "circumradius": float(R),
}
assert center_checks["incenter_max_side_distance_error"] < 1e-12
assert center_checks["circumcenter_max_vertex_radius_error"] < 1e-12

display_artifact(incenter_path, width=680)
center_checks

## Euler Line And Nine-Point Circle

For a non-equilateral triangle, the circumcenter `O`, centroid `G`, and orthocenter `H` lie on one line. The coordinate identity is sharper than a drawn straightedge: `H - G = 2(G - O)`. Once `H` is known, the nine-point circle is almost forced. Its center is the midpoint `N` of `O` and `H`; its radius is half the circumradius; and it passes through three side midpoints, three altitude feet, and three midpoints from the vertices to the orthocenter.

The static figure shows one triangle. The Plotly artifact after it samples many nearby triangles so you can zoom, hover, and see the same identity survive as the third vertex moves.

In [ ]:
H = orthocenter(BASE_TRIANGLE)
nine = nine_point_data(BASE_TRIANGLE)
N = nine["N"]
fig, ax = plt.subplots(figsize=(7.4, 6.0))
draw_clean_axes(ax, "Euler line and nine-point circle")
draw_triangle(ax, BASE_TRIANGLE, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
add_circle(ax, O, R, color=PALETTE["purple"], lw=1.5, ls="--")
add_circle(ax, N, R / 2, color=PALETTE["orange"], lw=2.0)
ax.plot([O[0], H[0]], [O[1], H[1]], color=PALETTE["red"], lw=1.8)
for name, P in nine["points"].items():
    color = PALETTE["orange"] if name.startswith("mid") else PALETTE["green"]
    ax.scatter(P[0], P[1], color=color, s=28, zorder=6)
for label, P, color in [("O", O, PALETTE["purple"]), ("G", G, PALETTE["blue"]), ("H", H, PALETTE["red"]), ("N", N, PALETTE["orange"]), ("I", I, PALETTE["green"])]:
    ax.scatter(P[0], P[1], color=color, s=62, zorder=7)
    ax.text(P[0] + 0.06, P[1] + 0.06, label, color=color, fontsize=11, weight="bold")
ax.text(-2.85, 3.22, "O, G, H are collinear; N=(O+H)/2", color=PALETTE["red"], fontsize=9)
ax.text(-2.85, 3.00, "Orange circle contains three families of points", color=PALETTE["orange"], fontsize=9)
ax.set_xlim(-3.1, 3.2); ax.set_ylim(-1.2, 3.55)
euler_path = save_figure(fig, artifact("figures", "euler-line-nine-point-circle.png"))

euler_checks = {
    "euler_collinearity_area": float(abs(cross2(G - O, H - O))),
    "euler_vector_identity_error": float(norm((H - G) - 2 * (G - O))),
    "nine_point_max_radius_error": float(max(nine["residuals"].values())),
    "nine_point_radius": float(R / 2),
}
assert euler_checks["euler_collinearity_area"] < 1e-12
assert euler_checks["euler_vector_identity_error"] < 1e-12
assert euler_checks["nine_point_max_radius_error"] < 1e-12

display_artifact(euler_path, width=700)
euler_checks

In [ ]:
experiment_rows = []
traces_by_center = {"O": [], "G": [], "H": [], "N": []}
A_fixed = np.array([0.15, 2.70]); B_fixed = np.array([-2.20, 0.05])
for t in np.linspace(0.0, 2 * math.pi, 72, endpoint=False):
    C_moved = np.array([2.10 + 0.55 * math.cos(t), 0.25 + 0.42 * math.sin(t)])
    tri = np.array([A_fixed, B_fixed, C_moved])
    if abs(triangle_area(tri)) < 1e-4:
        continue
    O_t, R_t = circumcenter(tri); G_t = centroid(tri); H_t = orthocenter(tri); N_t = midpoint(O_t, H_t)
    nine_t = nine_point_data(tri)
    row = {
        "t": float(t), "C_x": float(C_moved[0]), "C_y": float(C_moved[1]),
        "O_x": float(O_t[0]), "O_y": float(O_t[1]), "G_x": float(G_t[0]), "G_y": float(G_t[1]),
        "H_x": float(H_t[0]), "H_y": float(H_t[1]), "N_x": float(N_t[0]), "N_y": float(N_t[1]),
        "euler_area_error": float(abs(cross2(G_t - O_t, H_t - O_t))),
        "euler_vector_error": float(norm((H_t - G_t) - 2 * (G_t - O_t))),
        "nine_point_error": float(max(nine_t["residuals"].values())),
    }
    experiment_rows.append(row)
    for key, P in [("O", O_t), ("G", G_t), ("H", H_t), ("N", N_t)]:
        traces_by_center[key].append(P)

csv_path = write_csv(artifact("data", "euler-line-shape-experiment.csv"), experiment_rows)
fig_plotly = go.Figure()
colors = {"O": "#7c3aed", "G": "#2563eb", "H": "#dc2626", "N": "#d97706"}
for key, pts in traces_by_center.items():
    arr = np.array(pts)
    fig_plotly.add_trace(go.Scatter(x=arr[:, 0], y=arr[:, 1], mode="lines+markers", name=f"{key} locus", line=dict(color=colors[key], width=2), marker=dict(size=5), hovertemplate=f"{key}: (%{{x:.3f}}, %{{y:.3f}})<extra></extra>"))
for idx in range(0, len(experiment_rows), 12):
    row = experiment_rows[idx]
    O_t = np.array([row["O_x"], row["O_y"]]); H_t = np.array([row["H_x"], row["H_y"]]); C_t = np.array([row["C_x"], row["C_y"]])
    tri = np.array([A_fixed, B_fixed, C_t, A_fixed])
    fig_plotly.add_trace(go.Scatter(x=tri[:, 0], y=tri[:, 1], mode="lines", line=dict(color="rgba(31,41,55,0.28)", width=1), showlegend=False, hoverinfo="skip"))
    fig_plotly.add_trace(go.Scatter(x=[O_t[0], H_t[0]], y=[O_t[1], H_t[1]], mode="lines", line=dict(color="rgba(220,38,38,0.35)", width=1), showlegend=False, hoverinfo="skip"))
fig_plotly.update_layout(title="Euler line experiment: moving one vertex", xaxis=dict(scaleanchor="y", scaleratio=1, zeroline=False), yaxis=dict(zeroline=False), template="plotly_white", width=860, height=620, legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="left", x=0), margin=dict(l=30, r=30, t=80, b=30))
html_path = artifact("html", "euler-line-shape-experiment.html")
fig_plotly.write_html(html_path, include_plotlyjs=True, full_html=True)

euler_experiment_checks = {
    "samples": len(experiment_rows),
    "max_euler_area_error": float(max(row["euler_area_error"] for row in experiment_rows)),
    "max_euler_vector_error": float(max(row["euler_vector_error"] for row in experiment_rows)),
    "max_nine_point_error": float(max(row["nine_point_error"] for row in experiment_rows)),
}
assert euler_experiment_checks["samples"] >= 60
assert euler_experiment_checks["max_euler_vector_error"] < 1e-11
assert euler_experiment_checks["max_nine_point_error"] < 1e-11

display_artifact(html_path, width=820)
euler_experiment_checks

## Two Extremum Problems

The chapter's extremum problems turn a theorem into an optimization target. For an acute triangle, Fagnano's problem asks for the inscribed triangle of least perimeter; the orthic triangle, formed by the altitude feet, is the candidate. Fermat's problem asks for the interior point minimizing the sum of distances to the three vertices; when every angle is less than 120 degrees, the minimizing point sees the three vertices under three 120-degree angles.

The visual identifies each candidate, and the code tests whether a numerical optimizer lands at the same construction.

In [ ]:
F_a = foot(A, B, C); F_b = foot(B, C, A); F_c = foot(C, A, B)
orthic = np.array([F_a, F_b, F_c])
F, equilateral_vertices = fermat_point(BASE_TRIANGLE)
fig, axes = plt.subplots(1, 2, figsize=(12.3, 5.4))

ax = axes[0]
draw_clean_axes(ax, "Fagnano candidate: the orthic triangle")
draw_triangle(ax, BASE_TRIANGLE, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
draw_triangle(ax, orthic, labels=("U", "V", "W"), color=PALETTE["orange"], lw=2.2)
for V, foot_pt in [(A, F_a), (B, F_b), (C, F_c)]:
    ax.plot([V[0], foot_pt[0]], [V[1], foot_pt[1]], color=PALETTE["green"], lw=1.2, ls="--")
ax.text(-2.55, 2.83, "altitude feet define U,V,W", color=PALETTE["green"], fontsize=9)
ax.set_xlim(-2.85, 2.85); ax.set_ylim(-0.45, 3.15)

ax = axes[1]
draw_clean_axes(ax, "Fermat candidate: three 120-degree angles")
draw_triangle(ax, BASE_TRIANGLE, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
for side_points, label in [((B, C, equilateral_vertices["X_a"]), "X_a"), ((C, A, equilateral_vertices["X_b"]), "X_b"), ((A, B, equilateral_vertices["X_c"]), "X_c")]:
    P, Q, X = side_points; eq_tri = np.array([P, Q, X, P])
    ax.plot(eq_tri[:, 0], eq_tri[:, 1], color=PALETTE["purple"], lw=1.2, ls="--", alpha=0.85)
    ax.text(X[0] + 0.04, X[1] + 0.04, label, color=PALETTE["purple"], fontsize=8)
for V in BASE_TRIANGLE:
    ax.plot([F[0], V[0]], [F[1], V[1]], color=PALETTE["red"], lw=1.6)
ax.scatter(F[0], F[1], color=PALETTE["red"], s=64, zorder=6)
ax.text(F[0] + 0.07, F[1] + 0.07, "F", color=PALETTE["red"], fontsize=11, weight="bold")
add_angle_arc(ax, F, A, B, 0.30, PALETTE["red"], "120")
add_angle_arc(ax, F, B, C, 0.44, PALETTE["red"], "120")
add_angle_arc(ax, F, C, A, 0.58, PALETTE["red"], "120")
ax.set_xlim(-3.45, 3.95); ax.set_ylim(-4.15, 3.85)
extrema_path = save_figure(fig, artifact("figures", "extremum-problems-fagnano-fermat.png"))

def side_point(P: np.ndarray, Q: np.ndarray, t: float) -> np.ndarray:
    return (1 - t) * P + t * Q

def side_parameter(P: np.ndarray, Q: np.ndarray, X: np.ndarray) -> float:
    v = Q - P
    return float(np.dot(X - P, v) / np.dot(v, v))

def inscribed_perimeter(params: np.ndarray) -> float:
    U = side_point(B, C, params[0]); V = side_point(C, A, params[1]); W = side_point(A, B, params[2])
    return norm(U - V) + norm(V - W) + norm(W - U)

orthic_params = np.array([side_parameter(B, C, F_a), side_parameter(C, A, F_b), side_parameter(A, B, F_c)])
fagnano_result = minimize(inscribed_perimeter, x0=np.array([0.43, 0.54, 0.36]), bounds=[(0, 1)] * 3, method="L-BFGS-B")

def distance_sum(P: np.ndarray) -> float:
    return sum(norm(P - V) for V in BASE_TRIANGLE)

fermat_result = minimize(distance_sum, x0=G, method="BFGS")
fermat_angles = [math.degrees(angle_between(A - F, B - F)), math.degrees(angle_between(B - F, C - F)), math.degrees(angle_between(C - F, A - F))]
extremum_checks = {
    "orthic_perimeter": float(inscribed_perimeter(orthic_params)),
    "optimizer_perimeter": float(fagnano_result.fun),
    "orthic_parameter_error": float(norm(fagnano_result.x - orthic_params)),
    "fermat_distance_sum": float(distance_sum(F)),
    "optimizer_distance_sum": float(fermat_result.fun),
    "fermat_point_error": float(norm(fermat_result.x - F)),
    "fermat_angle_max_error_degrees": float(max(abs(angle - 120.0) for angle in fermat_angles)),
}
assert extremum_checks["optimizer_perimeter"] >= extremum_checks["orthic_perimeter"] - 1e-8
assert extremum_checks["orthic_parameter_error"] < 1e-4
assert extremum_checks["fermat_point_error"] < 1e-4
assert extremum_checks["fermat_angle_max_error_degrees"] < 1e-10

display_artifact(extrema_path, width=800)
extremum_checks

## Morley-Style Angle Trisectors

Morley's theorem is surprising because angle trisection is usually a warning sign: trisectors are hard to construct with straightedge and compass, and they do not behave like angle bisectors. The theorem says that the three intersections of adjacent trisectors form an equilateral triangle. The code constructs the two interior trisector rays at each vertex, intersects adjacent rays, and checks the three resulting side lengths.

In [ ]:
morley, trisector_rays = morley_triangle(BASE_TRIANGLE)
fig, ax = plt.subplots(figsize=(7.5, 6.3))
draw_clean_axes(ax, "Morley trisectors produce an equilateral triangle")
draw_triangle(ax, BASE_TRIANGLE, labels=("A", "B", "C"), color=PALETTE["ink"], lw=2.3, fill=True)
for V, label in zip(BASE_TRIANGLE, ["A", "B", "C"]):
    for ray in trisector_rays[label]:
        ray_unit = ray / norm(ray)
        ax.plot([V[0], V[0] + 3.3 * ray_unit[0]], [V[1], V[1] + 3.3 * ray_unit[1]], color=PALETTE["blue"], lw=1.0, alpha=0.62)
draw_triangle(ax, morley, labels=("P", "Q", "R"), color=PALETTE["orange"], lw=2.4)
ax.fill(morley[:, 0], morley[:, 1], color="#fef3c7", alpha=0.55, zorder=0)
ax.text(-2.65, 2.95, "Adjacent trisectors intersect at P, Q, R", color=PALETTE["blue"], fontsize=9)
ax.text(-2.65, 2.73, "The check measures PQ, QR, RP", color=PALETTE["orange"], fontsize=9)
ax.set_xlim(-2.9, 2.9); ax.set_ylim(-0.35, 3.25)
morley_path = save_figure(fig, artifact("figures", "morley-trisectors-equilateral.png"))

morley_lengths = [norm(morley[(i + 1) % 3] - morley[i]) for i in range(3)]
morley_angles = [math.degrees(angle_between(morley[(i - 1) % 3] - morley[i], morley[(i + 1) % 3] - morley[i])) for i in range(3)]
morley_checks = {
    "side_lengths": [float(x) for x in morley_lengths],
    "max_side_length_spread": float(max(morley_lengths) - min(morley_lengths)),
    "angle_degrees": [float(x) for x in morley_angles],
    "max_angle_error_degrees": float(max(abs(x - 60.0) for x in morley_angles)),
}
assert morley_checks["max_side_length_spread"] < 1e-11
assert morley_checks["max_angle_error_degrees"] < 1e-10

display_artifact(morley_path, width=700)
morley_checks

## Proof Dependency Map And Exact Checks

A proof-heavy chapter can feel like a list of named points unless the dependencies are tracked. The graph below records the route used in this notebook. It is not a substitute for proof, but it does show which constructions must already be trusted before later ones make sense.

The exact SymPy check complements the floating-point figures. For one rational triangle, SymPy solves for the circumcenter exactly and verifies the Euler-line vector identity without numerical rounding.

In [ ]:
G_proof = nx.DiGraph()
proof_edges = [
    ("primitive points/lines", "reflection isometry"), ("reflection isometry", "congruence tests"), ("congruence tests", "isosceles base angles"),
    ("midpoints", "medians"), ("medians", "centroid 2:1"), ("perpendicular bisectors", "circumcenter"),
    ("angle bisectors", "incenter"), ("altitudes", "orthocenter"), ("circumcenter", "Euler line"),
    ("centroid 2:1", "Euler line"), ("orthocenter", "Euler line"), ("Euler line", "nine-point circle"),
    ("altitude feet", "nine-point circle"), ("altitude feet", "Fagnano candidate"),
    ("equilateral side triangles", "Fermat candidate"), ("angle trisectors", "Morley triangle"),
]
G_proof.add_edges_from(proof_edges)
pos = nx.spring_layout(G_proof, seed=7, k=0.85)
fig, ax = plt.subplots(figsize=(10.5, 7.0))
draw_clean_axes(ax, "Dependency map for Chapter 01 constructions")
nx.draw_networkx_edges(G_proof, pos, ax=ax, arrowstyle="-|>", arrowsize=12, edge_color="#94a3b8", width=1.2)
nx.draw_networkx_nodes(G_proof, pos, ax=ax, node_size=1300, node_color="#eef2ff", edgecolors="#4338ca", linewidths=1.0)
nx.draw_networkx_labels(G_proof, pos, ax=ax, font_size=8.2, font_color=PALETTE["ink"])
proof_path = save_figure(fig, artifact("figures", "triangle-proof-dependency-map.png"))

A_s = sp.Matrix([sp.Rational(1, 2), sp.Rational(5, 2)]); B_s = sp.Matrix([-2, 0]); C_s = sp.Matrix([3, sp.Rational(1, 3)])
x, y = sp.symbols("x y"); O_s = sp.Matrix([x, y])
solution = sp.solve([
    sp.Eq((O_s - A_s).dot(O_s - A_s), (O_s - B_s).dot(O_s - B_s)),
    sp.Eq((O_s - A_s).dot(O_s - A_s), (O_s - C_s).dot(O_s - C_s)),
], [x, y], dict=True)[0]
O_s = sp.Matrix([sp.simplify(solution[x]), sp.simplify(solution[y])])
G_s = (A_s + B_s + C_s) / 3
H_s = A_s + B_s + C_s - 2 * O_s
symbolic_vector = sp.simplify((H_s - G_s) - 2 * (G_s - O_s))
symbolic_det = sp.simplify(sp.Matrix.hstack(G_s - O_s, H_s - O_s).det())
symbolic_checks = {
    "circumcenter": [str(value) for value in O_s], "centroid": [str(value) for value in G_s], "orthocenter": [str(value) for value in H_s],
    "euler_vector_identity": [str(value) for value in symbolic_vector], "euler_collinearity_determinant": str(symbolic_det),
}
assert symbolic_vector == sp.Matrix([0, 0])
assert symbolic_det == 0

display_artifact(proof_path, width=850)
symbolic_checks

In [ ]:
center_rows = [
    {"center": "centroid", "label": "G", "construction": "average of the three vertices", "invariant": "2:1 on every median", "x": float(G[0]), "y": float(G[1])},
    {"center": "incenter", "label": "I", "construction": "side-length weighted average", "invariant": "equal distances to the three sides", "x": float(I[0]), "y": float(I[1])},
    {"center": "circumcenter", "label": "O", "construction": "perpendicular bisectors", "invariant": "equal distances to the three vertices", "x": float(O[0]), "y": float(O[1])},
    {"center": "orthocenter", "label": "H", "construction": "altitudes", "invariant": "lies on Euler line with O and G", "x": float(H[0]), "y": float(H[1])},
    {"center": "nine-point center", "label": "N", "construction": "midpoint of O and H", "invariant": "nine-point radius is R/2", "x": float(N[0]), "y": float(N[1])},
    {"center": "Fermat point", "label": "F", "construction": "external equilateral triangles", "invariant": "three 120-degree angles", "x": float(F[0]), "y": float(F[1])},
]
center_table_path = write_csv(artifact("tables", "triangle-center-summary.csv"), center_rows)
all_figure_paths = [primitive_path, centroid_path, incenter_path, euler_path, extrema_path, morley_path, proof_path]
all_visual_paths = [*all_figure_paths, html_path]
all_check_groups = {
    "congruence": congruence_checks, "median_centroid": median_ratios, "incenter_circumcenter": center_checks,
    "euler_nine_point": euler_checks, "euler_experiment": euler_experiment_checks, "extrema": extremum_checks,
    "morley": morley_checks, "symbolic_euler": symbolic_checks,
}
checks_path = write_json(artifact("checks", "triangle-invariants.json"), {
    "chapter": CHAPTER_NO, "source_span": {"printed_pages": "3-25", "pdf_pages": "21-43"},
    "base_triangle": BASE_TRIANGLE.tolist(), "checks": all_check_groups,
})
visual_summary_path = write_json(artifact("checks", "visual_summary.json"), {
    "chapter": CHAPTER_NO, "source_span": {"printed_pages": "3-25", "pdf_pages": "21-43"},
    "visuals": [rel(path) for path in all_visual_paths], "figures": [rel(path) for path in all_figure_paths], "html": [rel(html_path)],
    "check_path": rel(checks_path), "data": [rel(csv_path)], "tables": [rel(center_table_path)],
    "storyboard_items": ["primitive concepts and congruence by reflection", "medians and centroid", "incenter and circumcenter", "Euler line and nine-point circle", "Fagnano and Fermat extremum checks", "Morley angle trisectors", "proof dependency map and exact Euler identity"],
})
manifest_rows = []
for path in [*all_visual_paths, csv_path, center_table_path, checks_path, visual_summary_path]:
    manifest_rows.append({"artifact": rel(path), "kind": path.parent.name, "bytes": path.stat().st_size})
manifest_path = write_csv(artifact("tables", "artifact-manifest.csv"), manifest_rows)

for path in [center_table_path, checks_path, visual_summary_path, manifest_path, csv_path]:
    display_artifact(path)
{"visuals": len(all_visual_paths), "checks": rel(checks_path), "manifest": rel(manifest_path)}

## Applied Lab: Move One Vertex, Watch What Survives

To use the notebook as a lab, change exactly one input in `BASE_TRIANGLE` and re-run the chapter. Keep the triangle nondegenerate and, for the Fagnano/Fermat section, keep it acute. Ask concrete questions: does the centroid still divide each median in the ratio 2:1, does the circumcenter move outside the triangle when the triangle becomes obtuse, does the Euler identity still pass, and does Morley's side-length spread remain near zero?

The next cell performs a deterministic perturbation without changing the main artifact triangle. It treats theorems as testable invariants rather than one-time drawings.

In [ ]:
perturbed = BASE_TRIANGLE.copy()
perturbed[2] += np.array([-0.35, 0.55])
O_p, R_p = circumcenter(perturbed); G_p = centroid(perturbed); H_p = orthocenter(perturbed)
nine_p = nine_point_data(perturbed); morley_p, _ = morley_triangle(perturbed)
morley_p_lengths = [norm(morley_p[(i + 1) % 3] - morley_p[i]) for i in range(3)]
lab_result = {
    "perturbed_C": [float(x) for x in perturbed[2]],
    "euler_vector_identity_error": float(norm((H_p - G_p) - 2 * (G_p - O_p))),
    "nine_point_max_radius_error": float(max(nine_p["residuals"].values())),
    "morley_side_length_spread": float(max(morley_p_lengths) - min(morley_p_lengths)),
}
assert lab_result["euler_vector_identity_error"] < 1e-11
assert lab_result["nine_point_max_radius_error"] < 1e-11
assert lab_result["morley_side_length_spread"] < 1e-10
lab_result

## Takeaways

- Congruence is visible only when the preservation claim is measured: reflection preserves side lengths and angles, and the isosceles base-angle result is one useful consequence.
- The centroid is affine, while the incenter and circumcenter are metric. Keeping those constructions separate prevents false intuition about triangle centers.
- The Euler line is best remembered as a vector identity, not as a fragile-looking straight line in one diagram.
- The nine-point circle packages nine different constructions into one radius check.
- Fagnano and Fermat show how classical geometry often hides an optimization problem behind a clean construction.
- Morley's theorem is a good warning against judging a construction by how unlikely it looks: the trisector triangle becomes equilateral anyway.

In [ ]:
final_artifacts = [*all_visual_paths, csv_path, center_table_path, manifest_path, checks_path, visual_summary_path]
assert_artifacts(final_artifacts, min_bytes=100)
numeric_thresholds = {
    "congruence_reflection": congruence_checks["max_reflected_side_length_error"],
    "centroid_ratio_A": abs(median_ratios["AG_over_GM"] - 2.0),
    "incenter_side_distance": center_checks["incenter_max_side_distance_error"],
    "circumcenter_vertex_radius": center_checks["circumcenter_max_vertex_radius_error"],
    "euler_vector_identity": euler_checks["euler_vector_identity_error"],
    "nine_point_radius": euler_checks["nine_point_max_radius_error"],
    "fagnano_parameter": extremum_checks["orthic_parameter_error"],
    "fermat_point": extremum_checks["fermat_point_error"],
    "morley_side_spread": morley_checks["max_side_length_spread"],
    "lab_euler_vector_identity": lab_result["euler_vector_identity_error"],
}
assert numeric_thresholds["congruence_reflection"] < 1e-12
assert numeric_thresholds["centroid_ratio_A"] < 1e-12
assert numeric_thresholds["incenter_side_distance"] < 1e-12
assert numeric_thresholds["circumcenter_vertex_radius"] < 1e-12
assert numeric_thresholds["euler_vector_identity"] < 1e-12
assert numeric_thresholds["nine_point_radius"] < 1e-12
assert numeric_thresholds["fagnano_parameter"] < 1e-4
assert numeric_thresholds["fermat_point"] < 1e-4
assert numeric_thresholds["morley_side_spread"] < 1e-11
assert numeric_thresholds["lab_euler_vector_identity"] < 1e-11

final_summary_path = artifact("checks", "final-sanity-summary.json")
write_json(final_summary_path, {"chapter": CHAPTER_NO, "status": "provisional"})
all_reported_artifacts = [*all_visual_paths, csv_path, center_table_path, manifest_path, checks_path, visual_summary_path, final_summary_path]
final_payload = {
    "chapter": CHAPTER_NO,
    "source_span": {"printed_pages": "3-25", "pdf_pages": "21-43"},
    "artifact_count": len(all_reported_artifacts),
    "numeric_thresholds": {key: float(value) for key, value in numeric_thresholds.items()},
    "symbolic_euler_vector_identity": symbolic_checks["euler_vector_identity"],
    "symbolic_euler_collinearity_determinant": symbolic_checks["euler_collinearity_determinant"],
}
for _ in range(10):
    sizes_before = {rel(path): path.stat().st_size for path in all_reported_artifacts}
    final_payload["artifact_sizes"] = sizes_before
    write_json(final_summary_path, final_payload)
    sizes_after_summary = {rel(path): path.stat().st_size for path in all_reported_artifacts}
    manifest_rows = [{"artifact": rel(path), "kind": path.parent.name, "bytes": sizes_after_summary[rel(path)]} for path in all_reported_artifacts]
    manifest_path = write_csv(artifact("tables", "artifact-manifest.csv"), manifest_rows)
    sizes_after_manifest = {rel(path): path.stat().st_size for path in all_reported_artifacts}
    if sizes_after_manifest == sizes_after_summary:
        break
assert_artifacts(all_reported_artifacts, min_bytes=100)

display_artifact(final_summary_path)
final_sanity = {
    "artifact_count": len(all_reported_artifacts),
    "largest_numeric_error": float(max(abs(value) for value in numeric_thresholds.values())),
    "final_summary": rel(final_summary_path),
}
final_sanity